In [0]:
# Cell 1 — Config
spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")

RAW_BUCKET = "s3://ecommerce-lakehouse-467091806172-raw-01"
BRONZE_BUCKET = "s3://ecommerce-lakehouse-467091806172-bronze-01"
SOURCE = "01_postgres"

# List what tables we have in raw
dbutils.fs.ls(f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=23/")

In [0]:
# Cell 2 — Create Bronze catalog with storage location
spark.sql("""
    CREATE CATALOG IF NOT EXISTS bronze
    MANAGED LOCATION 's3://ecommerce-lakehouse-467091806172-bronze-01'
""")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze.postgres")
spark.sql("SHOW SCHEMAS IN bronze")

In [0]:
# Cell 3 — Read orders from S3 Raw and write to Bronze Delta table
spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")

RAW_BUCKET = "s3://ecommerce-lakehouse-467091806172-raw-01"
SOURCE = "01_postgres"

# Read all orders parquet files across all dates
orders_raw = spark.read.format("parquet") \
    .load(f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=*/table=orders/")

print(f"Raw orders count: {orders_raw.count()}")
orders_raw.printSchema()

In [0]:
# Cell 4 — Write to Bronze Delta table
(orders_raw.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze.postgres.orders")
)

print("✅ bronze.postgres.orders written")
spark.sql("SELECT COUNT(*) as row_count FROM bronze.postgres.orders").show()
spark.sql("DESCRIBE TABLE bronze.postgres.orders").show(50)

In [0]:
# Cell 5 — Write remaining Postgres tables to Bronze
tables = ["customers", "payments", "order_items", "inventory"]

for table in tables:
    df = spark.read.format("parquet") \
        .load(f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=*/table={table}/")
    
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"bronze.postgres.{table}")
    
    count = spark.sql(f"SELECT COUNT(*) as cnt FROM bronze.postgres.{table}").collect()[0]['cnt']
    print(f"✅ bronze.postgres.{table}: {count} rows")

In [0]:
# Cell 6 — Verify all tables in Unity Catalog
spark.sql("SHOW TABLES IN bronze.postgres").show()